# Incremental Dataset Generator

Creates update files that continue after each original dataset:

| Dataset | Output | Requirements |
| --- | --- | --- |
| Taxi trips | Parquet | 5–10% new trips after latest pickup; resemble originals; 1–2% exact duplicates from source |
| Weather | CSV | New hourly rows after latest observation; add `humidity` (20–100%) |
| Air quality | CSV | New hourly rows after latest observation; add `aqi` (0–500) |


### Helper functions


In [ ]:
from collections import deque
import csv
from datetime import datetime, timedelta
import io
import math
import os
import random
import shutil
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def _estimate_csv_row_count(source_path: str) -> int:
    """Estimate data rows from file size / average line length (no full scan)."""
    file_size = os.path.getsize(source_path)
    if file_size == 0:
        return 0
    with open(source_path, "rb") as source_file:
        sample = source_file.read(1 << 20)
    newline_count = sample.count(b"\n") or 1
    avg_line_length = len(sample) / newline_count
    return max(0, int(file_size / avg_line_length) - 1)


def _read_csv_tail_values(source_path: str, n_rows: int, header: list) -> list:
    """Read the last n_rows data lines by seeking from EOF."""
    if n_rows <= 0:
        return []

    with open(source_path, "rb") as source_file:
        source_file.seek(0, os.SEEK_END)
        file_size = source_file.tell()
        if file_size == 0:
            return []

        target_lines = n_rows + 2
        block_size = 1 << 20
        buffer = b""
        position = file_size

        while position > 0 and buffer.count(b"\n") < target_lines:
            read_size = min(block_size, position)
            position -= read_size
            source_file.seek(position)
            buffer = source_file.read(read_size) + buffer

    text = buffer.decode("utf-8-sig", errors="replace")
    lines = text.splitlines()
    if position > 0 and lines:
        lines = lines[1:]
    if lines and lines[0].startswith(header[0].lstrip("\ufeff")):
        lines = lines[1:]

    data_lines = lines[-n_rows:] if len(lines) >= n_rows else lines
    if not data_lines:
        return []

    reader = csv.reader(io.StringIO("\n".join(data_lines) + "\n"))
    return [row for row in reader if row]


def inspect_source_csv_by_date(source_path: str, update_fraction: float, datetime_builder):
    """
    Load a small EOF window and keep rows in the last
    (8760 * update_fraction) hours before the latest timestamp found there.
    Avoids two full scans of multi-GB CSVs.
    """
    with open(source_path, "r", newline="", encoding="utf-8-sig") as source_file:
        source_header = next(csv.reader(source_file))

    source_row_count = _estimate_csv_row_count(source_path)
    target_size = max(1, math.ceil(source_row_count * update_fraction))
    read_size = max(target_size, math.ceil(source_row_count * min(0.05, update_fraction * 5)))

    tail_values = _read_csv_tail_values(source_path, read_size, source_header)
    source_tail = [dict(zip(source_header, values)) for values in tail_values]

    if not source_tail:
        buffer = deque(maxlen=read_size)
        with open(source_path, "r", newline="", encoding="utf-8-sig") as source_file:
            reader = csv.reader(source_file)
            next(reader)
            counted = 0
            for values in reader:
                counted += 1
                buffer.append(values)
        source_row_count = counted
        source_tail = [dict(zip(source_header, values)) for values in buffer]

    dated_rows = []
    for row in source_tail:
        row_dt = datetime_builder(row)
        if row_dt is not None:
            dated_rows.append((row_dt, row))

    if not dated_rows:
        return source_header, source_row_count, [], None, None

    latest_dt = max(dt for dt, _ in dated_rows)
    target_hours = max(1, math.ceil(8760 * update_fraction))
    cutoff_dt = latest_dt - timedelta(hours=target_hours)
    source_tail = [row for dt, row in dated_rows if dt > cutoff_dt]

    if not source_tail:
        source_tail = [row for dt, row in dated_rows if dt == latest_dt]
        cutoff_dt = latest_dt

    return source_header, source_row_count, source_tail, latest_dt, cutoff_dt


def write_incremental_csv(update_path: str, update_header: list, rows: list):
    os.makedirs(os.path.dirname(update_path) or ".", exist_ok=True)
    with open(update_path, "w", newline="", encoding="utf-8") as update_file:
        writer = csv.DictWriter(update_file, fieldnames=update_header, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def _jitter_numeric(value, scale=0.03, min_value=None, max_value=None):
    """Small relative noise so shifted copies are realistic but not identical."""
    try:
        number = float(value)
    except (TypeError, ValueError):
        return value
    if math.isnan(number):
        return value
    jittered = number * (1.0 + random.uniform(-scale, scale))
    if min_value is not None:
        jittered = max(min_value, jittered)
    if max_value is not None:
        jittered = min(max_value, jittered)
    return jittered


def generate_incremental_csv_update(
    source_path: str,
    update_path: str,
    update_fraction: float,
    datetime_builder,
    datetime_formatter,
    transform_row=None,
    extra_columns: list = None,
    time_step: timedelta = timedelta(hours=1),
    numeric_jitter_columns: dict = None,
):
    """
    Time-shift the last update_fraction window so new rows start immediately
    after the latest observation. Optional transform_row adds schema-evolution columns.
    """
    (
        source_header,
        source_row_count,
        source_tail,
        latest_dt,
        cutoff_dt,
    ) = inspect_source_csv_by_date(source_path, update_fraction, datetime_builder)

    if not source_tail or latest_dt is None:
        raise ValueError(f"No valid records or timestamps found in {source_path}")

    target_start_dt = latest_dt + time_step
    time_shift = target_start_dt - cutoff_dt - time_step
    jitter_cols = numeric_jitter_columns or {}

    generated_rows = []
    for source_row in source_tail:
        row_dict = dict(source_row)
        orig_dt = datetime_builder(row_dict)

        if orig_dt is not None:
            new_dt = orig_dt + time_shift
            row_dict = datetime_formatter(row_dict, new_dt, time_shift)

        for col, bounds in jitter_cols.items():
            if col in row_dict and row_dict[col] not in (None, "", "NULL"):
                lo, hi = bounds
                jittered = _jitter_numeric(row_dict[col], min_value=lo, max_value=hi)
                # Keep CSV-friendly formatting
                if isinstance(jittered, float):
                    row_dict[col] = f"{jittered:.3f}".rstrip("0").rstrip(".")
                else:
                    row_dict[col] = jittered

        if transform_row is not None:
            row_dict = transform_row(row_dict)

        generated_rows.append(row_dict)

    update_header = list(source_header) + list(extra_columns or [])
    write_incremental_csv(update_path, update_header, generated_rows)

    period_start = datetime_builder(generated_rows[0])
    period_end = datetime_builder(generated_rows[-1])
    return source_row_count, len(generated_rows), period_start, period_end


def generate_taxi_trips_update(
    source_path: str,
    update_path: str,
    new_fraction: float = 0.07,
    duplicate_fraction: float = 0.015,
    seed: int = 42,
):
    """
    Build a Parquet update for taxi trips:
    - new_fraction (5–10%) sampled trips with timestamps shifted after the latest pickup
    - duplicate_fraction (1–2%) exact copies from the original dataset
    """
    from pyspark.sql import functions as F
    from src.spark import create_spark

    if not (0.05 <= new_fraction <= 0.10):
        raise ValueError("new_fraction must be between 0.05 and 0.10")
    if not (0.01 <= duplicate_fraction <= 0.02):
        raise ValueError("duplicate_fraction must be between 0.01 and 0.02")

    spark = create_spark("data-generator-taxi")
    trips = spark.read.parquet(source_path)

    # Ignore obviously corrupt historical timestamps when finding the true end
    valid = trips.filter(
        (F.col("tpep_pickup_datetime") >= F.lit("2024-01-01").cast("timestamp"))
        & (F.col("tpep_pickup_datetime") < F.lit("2025-01-01").cast("timestamp"))
    )

    source_row_count = valid.count()
    latest_pickup = valid.agg(F.max("tpep_pickup_datetime")).collect()[0][0]
    if latest_pickup is None:
        spark.stop()
        raise ValueError(f"No valid pickup timestamps in {source_path}")

    templates = valid.sample(withReplacement=False, fraction=new_fraction, seed=seed).cache()
    template_count = templates.count()
    min_template_pickup = templates.agg(F.min("tpep_pickup_datetime")).collect()[0][0]
    # Shift so EVERY new trip is after the original latest (anchor on sample minimum)
    shift_seconds = int((latest_pickup - min_template_pickup).total_seconds()) + 3600

    # TIMESTAMP_NTZ cannot cast directly to long — use unix seconds via timestamp
    new_trips = (
        templates
        .withColumn(
            "tpep_pickup_datetime",
            (
                F.unix_timestamp(F.col("tpep_pickup_datetime").cast("timestamp"))
                + F.lit(shift_seconds)
            ).cast("timestamp"),
        )
        .withColumn(
            "tpep_dropoff_datetime",
            (
                F.unix_timestamp(F.col("tpep_dropoff_datetime").cast("timestamp"))
                + F.lit(shift_seconds)
            ).cast("timestamp"),
        )
        # Light noise on distance/fare so values resemble but are not byte-identical
        .withColumn(
            "trip_distance",
            F.round(F.col("trip_distance") * (F.lit(1.0) + (F.rand(seed) - 0.5) * 0.06), 2),
        )
        .withColumn(
            "fare_amount",
            F.round(F.col("fare_amount") * (F.lit(1.0) + (F.rand(seed + 1) - 0.5) * 0.06), 2),
        )
        .withColumn(
            "total_amount",
            F.round(F.col("total_amount") * (F.lit(1.0) + (F.rand(seed + 2) - 0.5) * 0.06), 2),
        )
    )

    n_duplicates = max(1, int(math.ceil(source_row_count * duplicate_fraction)))
    duplicates = valid.orderBy(F.rand(seed + 3)).limit(n_duplicates)

    update_df = new_trips.unionByName(duplicates)

    out = Path(update_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = out.parent / f".tmp_{out.stem}"
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    if out.exists():
        if out.is_dir():
            shutil.rmtree(out)
        else:
            out.unlink()

    (
        update_df.coalesce(1)
        .write.mode("overwrite")
        .parquet(str(tmp_dir))
    )

    part_files = list(tmp_dir.glob("*.parquet"))
    if not part_files:
        spark.stop()
        raise RuntimeError(f"Spark wrote no parquet parts under {tmp_dir}")
    shutil.move(str(part_files[0]), str(out))
    shutil.rmtree(tmp_dir)

    new_min, new_max = new_trips.agg(
        F.min("tpep_pickup_datetime"), F.max("tpep_pickup_datetime")
    ).collect()[0]

    templates.unpersist()
    spark.stop()
    return (
        source_row_count,
        template_count,
        n_duplicates,
        latest_pickup,
        new_min,
        new_max,
        str(out),
    )


### Taxi trips

Parquet update: **7%** new trips (within 5–10%) after the latest pickup, plus **1.5%** exact duplicates from the original data (within 1–2%).


In [ ]:
source_path = str(ROOT / "data/raw/taxi_trips/yellow")
update_path = str(ROOT / "data/raw/taxi_trips/yellow_tripdata_update.parquet")

print("Generating taxi trips incremental update...")
(
    source_row_count,
    new_records,
    duplicates,
    latest_pickup,
    period_start,
    period_end,
    out_path,
) = generate_taxi_trips_update(
    source_path=source_path,
    update_path=update_path,
    new_fraction=0.07,          # mid-point of 5–10%
    duplicate_fraction=0.015,   # mid-point of 1–2%
)

print("=== TAXI TRIPS GENERATION REPORT ===")
print(f"Source directory:       {source_path}")
print(f"Source records:         {source_row_count:,}")
print(f"Output update file:     {out_path}")
print(f"New records written:    {new_records:,} ({100 * new_records / source_row_count:.2f}% of source)")
print(f"Duplicates injected:    {duplicates:,} ({100 * duplicates / source_row_count:.2f}% of source)")
print(f"Total update records:   {(new_records + duplicates):,}")
print(f"Original latest pickup: {latest_pickup}")
print(f"New trip period:        {period_start} to {period_end}")


### Weather

CSV update for the hours immediately after the original series, with schema evolution column `humidity` (relative humidity %, clipped to 20–100).


In [ ]:
from datetime import datetime, timedelta


def weather_dt_builder(row: dict):
    try:
        return datetime(
            int(row["year"]),
            int(row["month"]),
            int(row["day"]),
            int(float(row["hour"])),
        )
    except (KeyError, ValueError, TypeError):
        return None


def weather_dt_formatter(row: dict, new_dt: datetime, time_shift: timedelta) -> dict:
    row["year"] = str(new_dt.year)
    row["month"] = str(new_dt.month)
    row["day"] = str(new_dt.day)
    row["hour"] = str(new_dt.hour)
    return row


def weather_transform(row: dict) -> dict:
    """Add humidity in [20, 100], derived from rhum when present."""
    raw = row.get("rhum")
    try:
        humidity = float(raw)
    except (TypeError, ValueError):
        humidity = random.uniform(40.0, 85.0)
    humidity = humidity * (1.0 + random.uniform(-0.02, 0.02))
    humidity = min(100.0, max(20.0, humidity))
    row["humidity"] = f"{humidity:.1f}"
    return row


source_path = str(ROOT / "data/raw/weather/weather.csv")
update_path = str(ROOT / "data/raw/weather/weather_update.csv")

source_row_count, new_records, start_period, end_period = generate_incremental_csv_update(
    source_path=source_path,
    update_path=update_path,
    update_fraction=0.01,  # ~88 new hourly observations after latest
    datetime_builder=weather_dt_builder,
    datetime_formatter=weather_dt_formatter,
    transform_row=weather_transform,
    extra_columns=["humidity"],
    numeric_jitter_columns={
        "temp": (-50.0, 50.0),
        "wspd": (0.0, 80.0),
        "prcp": (0.0, 50.0),
        "pres": (950.0, 1050.0),
        "rhum": (20.0, 100.0),
    },
)

print("=== WEATHER GENERATION REPORT ===")
print(f"Source file:            {source_path}")
print(f"Source records:         {source_row_count:,}")
print(f"Output update file:     {update_path}")
print(f"New records written:    {new_records:,}")
print(f"Schema Evolution:       Added column 'humidity' (relative humidity %, 20–100)")
print(f"Time period covered:    {start_period} to {end_period}")


### Air quality

CSV update for the hours immediately after the original series, with schema evolution column `aqi` (Air Quality Index, clipped to 0–500) derived from PM2.5 via EPA breakpoints.


In [ ]:
from datetime import datetime, timedelta

aqi_breakpoints = [
    (0.0, 9.0, 0, 50),
    (9.1, 35.4, 51, 100),
    (35.5, 55.4, 101, 150),
    (55.5, 125.4, 151, 200),
    (125.5, 225.4, 201, 300),
    (225.5, 500.4, 301, 500),
]


def pm25_to_aqi(measurement) -> str:
    if measurement is None or str(measurement).strip() == "":
        return ""
    try:
        concentration = max(0.0, float(measurement))
    except (TypeError, ValueError):
        return ""

    for c_lo, c_hi, a_lo, a_hi in aqi_breakpoints:
        if concentration <= c_hi:
            aqi = ((a_hi - a_lo) / (c_hi - c_lo)) * (concentration - c_lo) + a_lo
            return str(int(min(500, max(0, round(aqi)))))
    return "500"


def air_quality_dt_builder(row):
    try:
        return datetime.strptime(
            f"{row['Date Local']} {row['Time Local']}", "%Y-%m-%d %H:%M"
        )
    except (KeyError, ValueError):
        return None


def air_quality_dt_formatter(row, new_dt, time_shift):
    row["Date Local"] = new_dt.strftime("%Y-%m-%d")
    row["Time Local"] = new_dt.strftime("%H:%M")
    if row.get("Date GMT") and row.get("Time GMT"):
        try:
            orig_gmt = datetime.strptime(
                f"{row['Date GMT']} {row['Time GMT']}", "%Y-%m-%d %H:%M"
            )
            new_gmt = orig_gmt + time_shift
            row["Date GMT"] = new_gmt.strftime("%Y-%m-%d")
            row["Time GMT"] = new_gmt.strftime("%H:%M")
        except ValueError:
            pass
    return row


def air_quality_transform(row):
    # Optional light noise on the measurement, then derive AQI in [0, 500]
    if row.get("Sample Measurement") not in (None, ""):
        jittered = _jitter_numeric(row["Sample Measurement"], scale=0.05, min_value=0.0)
        if isinstance(jittered, float):
            row["Sample Measurement"] = f"{jittered:.3f}".rstrip("0").rstrip(".")
    row["aqi"] = pm25_to_aqi(row.get("Sample Measurement", ""))
    return row


source_path = str(ROOT / "data/raw/air_quality/hourly_88101_2024.csv")
update_path = str(ROOT / "data/raw/air_quality/hourly_88101_update.csv")

print("Generating air quality incremental update...")
source_row_count, new_records, start_period, end_period = generate_incremental_csv_update(
    source_path=source_path,
    update_path=update_path,
    update_fraction=0.01,
    datetime_builder=air_quality_dt_builder,
    datetime_formatter=air_quality_dt_formatter,
    transform_row=air_quality_transform,
    extra_columns=["aqi"],
)

print("=== AIR QUALITY GENERATION REPORT ===")
print(f"Source file:            {source_path}")
print(f"Source records:         {source_row_count:,}")
print(f"Output update file:     {update_path}")
print(f"New records written:    {new_records:,}")
print(f"Schema Evolution:       Added column 'aqi' (Air Quality Index, 0–500)")
print(f"Time period covered:    {start_period} to {end_period}")
